In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
os.getcwd()

'/home/dermodkkelly/rumen_microbiome_pipeline/postprocessing'

In [3]:
os.chdir("/home/dermodkkelly/rumen_microbiome_pipeline")


In [4]:
FEATURE_FILE = "results/rumen_combined_feature-table.tsv"
TAX_FILE = "results/rumen_combined_exported-taxonomy.tsv"

OUTDIR = Path("postprocessing/output")
OUTDIR.mkdir(parents=True, exist_ok=True)


In [5]:
tax = pd.read_csv(TAX_FILE, sep="\t")
tax.head()

,Feature ID,Taxon,Confidence
0,0000b84a296311cfe214483f9e2bfe28,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.838961
1,000135ece78204cf880cf0c64a4f6cb7,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.937315
2,0001a1fb4083eb04593b5d4e110c31ba,d__Bacteria; p__Bacteroidota; c__Bacteroidia,0.986549
3,0001fb18436a5c8e4c11d91e9aa3c6e9,d__Bacteria; p__Actinobacteriota; c__Coriobact...,0.988822
4,00021929c0a6e082fd353a3d9fef8e51,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.960691


In [6]:
def parse_taxonomy(t):
    ranks = {"Kingdom":"Unassigned","Phylum":"Unassigned","Class":"Unassigned",
             "Order":"Unassigned","Family":"Unassigned","Genus":"Unassigned","Species":"Unassigned"}

    if pd.isna(t):
        return pd.Series(ranks)

    parts = [x.strip() for x in str(t).split(";")]

    rank_map = {
        "d__":"Kingdom",
        "p__":"Phylum",
        "c__":"Class",
        "o__":"Order",
        "f__":"Family",
        "g__":"Genus",
        "s__":"Species"
    }

    for p in parts:
        for prefix, rank in rank_map.items():
            if p.startswith(prefix):
                val = p.replace(prefix, "").strip()
                if val == "":
                    val = "Unassigned"
                ranks[rank] = val

    return pd.Series(ranks)

parsed = tax["Taxon"].apply(parse_taxonomy)
tax = pd.concat([tax, parsed], axis=1)

tax.head()

,Feature ID,Taxon,Confidence,Kingdom,Phylum,Class,Order,Family,Genus,Species
0,0000b84a296311cfe214483f9e2bfe28,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.838961,Bacteria,Firmicutes,Clostridia,Christensenellales,Christensenellaceae,Christensenellaceae_R-7_group,uncultured_rumen
1,000135ece78204cf880cf0c64a4f6cb7,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.937315,Bacteria,Firmicutes,Clostridia,Christensenellales,Christensenellaceae,Christensenellaceae_R-7_group,Unassigned
2,0001a1fb4083eb04593b5d4e110c31ba,d__Bacteria; p__Bacteroidota; c__Bacteroidia,0.986549,Bacteria,Bacteroidota,Bacteroidia,Unassigned,Unassigned,Unassigned,Unassigned
3,0001fb18436a5c8e4c11d91e9aa3c6e9,d__Bacteria; p__Actinobacteriota; c__Coriobact...,0.988822,Bacteria,Actinobacteriota,Coriobacteriia,Coriobacteriales,Unassigned,Unassigned,Unassigned
4,00021929c0a6e082fd353a3d9fef8e51,d__Bacteria; p__Firmicutes; c__Clostridia; o__...,0.960691,Bacteria,Firmicutes,Clostridia,Peptostreptococcales-Tissierellales,Anaerovoracaceae,Family_XIII_AD3011_group,Unassigned


In [7]:
for rank in ["Phylum","Class","Order","Family","Genus","Species"]:
    assigned = (tax[rank] != "Unassigned").sum()
    pct = assigned / len(tax) * 100
    print(rank, assigned, round(pct,1))

Phylum 79395 94.3
Class 78633 93.4
Order 76620 91.0
Family 75016 89.1
Genus 67256 79.8
Species 46222 54.9


In [8]:
feat = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    skiprows=1
)

feat.rename(columns={feat.columns[0]: "FeatureID"}, inplace=True)
feat.head()

,FeatureID,Tully__10732_S4,Tully__10777_S5,Tully__10785_S6,Tully__11203_S10,Tully__11308_S12,Tully__20270_S14,Tully__20412_S16,Tully__21085_S22,Tully__21093_S24,...,EN00011679__Sheep_CT24_90,EN00011679__Sheep_CT24_91,EN00011679__Sheep_CT24_92,EN00011679__Sheep_CT24_93,EN00011679__Sheep_CT24_94,EN00011679__Sheep_CT24_95,EN00011679__Sheep_CT24_96,EN00011679__Sheep_CT24_97,EN00011679__Sheep_CT24_98,EN00011679__Sheep_CT24_99
0,0000b84a296311cfe214483f9e2bfe28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,000135ece78204cf880cf0c64a4f6cb7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0001a1fb4083eb04593b5d4e110c31ba,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0001fb18436a5c8e4c11d91e9aa3c6e9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,00021929c0a6e082fd353a3d9fef8e51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
feat = feat.set_index("FeatureID")

sample_matrix = feat.T
sample_matrix.index.name = "SampleID"
sample_matrix.reset_index(inplace=True)

sample_matrix.head()

FeatureID,SampleID,0000b84a296311cfe214483f9e2bfe28,000135ece78204cf880cf0c64a4f6cb7,0001a1fb4083eb04593b5d4e110c31ba,0001fb18436a5c8e4c11d91e9aa3c6e9,00021929c0a6e082fd353a3d9fef8e51,00025030599fa2848af2711196b72760,00034ec4df8628c90b5669ba90db0def,000407eb2b7635995cd64f936d5753d1,00040bd2e1f507e209fb0a153a6ca3a0,...,fff9feea247cb2ff1b24576006e1f44b,fffccadc8fcf99b306bd54d18c203d0c,fffd1275c99241770dbdecb31991d3dd,fffd3a11c88e827f4377bca16665f989,fffdaf7af517e4e6fcae31b6c54091ae,fffdc059674592831636fd03baa29eed,fffe1366203f43d251526329507be2d1,fffeda2f54ff42633e5926721bc88de8,ffff5b76e7c8c74f71dfb6b6da89c7a6,ffffa5fc5e60f992857e2d63814d3452
0,Tully__10732_S4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Tully__10777_S5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Tully__10785_S6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Tully__11203_S10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Tully__11308_S12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# ---- genus lookup ----
lookup = tax[["Feature ID","Genus"]].copy()
lookup.columns = ["FeatureID","Genus"]
lookup["Genus"] = lookup["Genus"].replace("", "Unassigned")

# ---- collapse ASVs to genus (fast column-grouping, no giant melt) ----
# feat is features x samples, indexed by FeatureID
feat_genus = feat.copy()
feat_genus.index = feat_genus.index.map(
    lookup.set_index("FeatureID")["Genus"]
).fillna("Unassigned")

genus_by_sample = feat_genus.groupby(level=0).sum()      # rows = genus, cols = samples
genus_wide = genus_by_sample.T.reset_index().rename(columns={"index": "SampleID"})

genus_wide.head()

FeatureID,SampleID,0319-6G20,0319-7L14,11-24,1174-901-12,67-14,A4b,ADurb.Bin063-1,ASF356,Abditibacterium,...,mle1-7,p-1088-a5_gut_group,p-251-o5,p-2534-18B5_gut_group,possible_genus_Sk018,probable_genus_10,uncultured,vadinBA26,vadinBE97,vadinHA49
0,Tully__10732_S4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,238.0,45.0,0.0,0.0,36.0,1119.0,0.0,91.0,0.0
1,Tully__10777_S5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,144.0,160.0,0.0,0.0,349.0,1265.0,0.0,75.0,0.0
2,Tully__10785_S6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,119.0,191.0,0.0,0.0,160.0,3133.0,0.0,206.0,0.0
3,Tully__11203_S10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,29.0,0.0,2.0,0.0,0.0,627.0,0.0,11.0,0.0
4,Tully__11308_S12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,179.0,24.0,3.0,0.0,273.0,2140.0,0.0,29.0,0.0


In [11]:
print(genus_wide.shape)          
print(genus_wide["SampleID"].nunique())   # expect 1487, all unique
# genus matrix row sums should equal the original per-sample depth
genus_wide.set_index("SampleID").sum(axis=1).describe()

(1761, 859)
1761


count      1761.000000
mean      47663.461670
std       17544.248946
min           0.000000
25%       36145.000000
50%       42852.000000
75%       56673.000000
max      139074.000000
dtype: float64

In [12]:
# what are the lowest-depth samples?
depths = genus_wide.set_index("SampleID").sum(axis=1).sort_values()
depths.head(15)

SampleID
Tully__60424_S64                 0.0
EN00011689__Minus17             33.0
Tully__80322_S37                52.0
Tully__80503_S55                69.0
EN00011686__Minus19            568.0
EN00011684__Minus16            603.0
EN00011679__ControlN_ive_6     719.0
EN00011687__ControlN_ive_5     732.0
EN00010710__N1                1171.0
EN00010710__N3                1478.0
EN00010710__N2                2080.0
EN00011685__Minus16           2322.0
EN00011679__Minus1            2603.0
EN00011682__Minus14           2690.0
EN00011681__Minus5            2770.0
dtype: float64

In [13]:
print("samples < 1000 reads:", (depths < 1000).sum())
print("samples < 5000 reads:", (depths < 5000).sum())

samples < 1000 reads: 8
samples < 5000 reads: 17


In [14]:
import re

# corrected: allow optional underscore before trailing number (handles ControlP_ive_5)
control_pattern = r'__(?:ControlN_ive|ControlNive|ControlP_ive|Minus|Plus|N|P)_?\d*$'
is_control = genus_wide["SampleID"].str.contains(control_pattern, regex=True)

print("Flagged as control/blank:", is_control.sum())      # expect 28
print(sorted(genus_wide.loc[is_control, "SampleID"].tolist()))

Flagged as control/blank: 28
['EN00010710__N1', 'EN00010710__N2', 'EN00010710__N3', 'EN00010710__N4', 'EN00010710__N5', 'EN00010710__P1', 'EN00011679__ControlN_ive_6', 'EN00011679__ControlP_ive_5', 'EN00011679__Minus1', 'EN00011679__Plus1', 'EN00011681__ControlNive_8', 'EN00011681__Minus5', 'EN00011681__Plus4', 'EN00011682__ControlP_ive_8', 'EN00011682__Minus14', 'EN00011682__Plus5', 'EN00011684__ControlP_ive_9', 'EN00011684__Minus16', 'EN00011684__Plus14', 'EN00011685__ControlP_ive_4', 'EN00011685__Minus16', 'EN00011685__Plus16', 'EN00011686__Minus19', 'EN00011686__Plus19', 'EN00011687__ControlN_ive_5', 'EN00011687__ControlP_ive_10', 'EN00011689__Minus17', 'EN00011689__Plus17']


In [15]:
# Save corrected genus-level count tables
genus_wide.to_csv(
    OUTDIR / "genus_counts_with_controls_515F_806R.csv",
    index=False
)

genus_bio = genus_wide[~is_control].reset_index(drop=True)

genus_bio.to_csv(
    OUTDIR / "genus_counts_no_controls_515F_806R.csv",
    index=False
)

print(f"with_controls: {genus_wide.shape[0]}")   # expect 1487
print(f"no_controls:   {genus_bio.shape[0]}")    # expect 1459

# Save corrected taxonomy lookup
tax.to_csv(
    OUTDIR / "taxonomy_lookup_515F_806R.csv",
    index=False
)

with_controls: 1761
no_controls:   1733
